In [ ]:
# Contains sets of code snippets that run various sweeps, maybe even feedback control in future...
import os
import pitayaconfig
import cooling
if os.name == "posix": # Force wayland-qt compatibility.
    os.environ["QT_QPA_PLATFORM"] = "xcb"
import pyrpl

In [ ]:
p = pyrpl.Pyrpl(hostname=pitayaconfig.HOSTNAME1, config="",reloadfpga=True,gui=True)
#pyrpl.APP.exec_()

In [ ]:
# Setup script that sets the iq, squaring, PID, parameters instead of using the configs as they're notoriously buggy.
# Route the signal generated by IQ0 to the Red Pitaya's analog output 1

# Define 
iq0 = p.rp.iq0     # IQ modules
iq1 = p.rp.iq1
sq0 = p.rp.square0 # Squaring modules
sq1 = p.rp.square1
pid0 = p.rp.pid0   #pid modules
pid1 = p.rp.pid1

# Rout iq output to squaring modules
sq0.input = 'iq0'
sq1.input = 'iq1'

# Rout squaring output to pid modules
pid0.input = 'square0'


# Initialise the Each modules with sensible starting parameters
FREQ1 = 42000 # Centre frequency of the iq0 module
FREQ2 = 165000 # Centre frequency of the iq1 module
AC_COUPLING_BANDWIDTH_HZ = 5000 # Longapass frequency (12db/oct)
BW1 = 1000 # Bandwidth of the iq0 module output.
BW2 = 2000 # Bandwidth of the iq1 module output.

iq0.setup(frequency = FREQ1,
         bandwidth = [BW1]*2,
         gain = 0,
         phase = 0,
         acbandwidth = AC_COUPLING_BANDWIDTH_HZ,
         amplitude = 0,
         input = 'in1',
         output_signal = 'output_direct',
         modulation_at_2f = 'off',
         demodulation_at_2f = 'off')

iq1.setup(frequency = FREQ2,
         bandwidth = [BW2]*2,
         gain = 0,
         phase = 0,
         acbandwidth = AC_COUPLING_BANDWIDTH_HZ,
         amplitude = 0,
         input = 'in2',
         output_signal = 'output_direct',
         modulation_at_2f = 'off',
         demodulation_at_2f = 'off')

# Set the pid parameters
pid0.setup(proportional = 1,
            integral = 0,
            input = 'square0',
            derivative = 0,
            inputfilter = [-FREQ2/2]*4, # Cut off the residual fundamental from the squaring step
            outputdirect = 'out1')

pid1.setup(proportional = 1,
            integral = 0,
            input = 'square1',
            derivative = 0,
            inputfilter = [-FREQ2/2]*4, # Cut off the residual fundamental from the squaring module
            outputdirect = 'out2')

# Also setup the spectrum analyzer and the oscilloscope to monitor the signals
spec = p.spectrumanalyzer
scope = p.rp.scope

spec.setup(input1_baseband = 'in1', #Can try out2 as well.
            input2_baseband = 'out1',
            span = 2E5,
            display_cross_amplitude = 'off',, 
            averaging = 'on', 
            averagecount = 10)

scope.setup(input1 = 'in1',
            input2 = 'out1'
            duration = 0.001
            trigger_source = 'in1',)

In [ ]:
# Ideas?
# Run frequency sweep on IQ module
# Run phase sweep on IQ module, recording thespectrum at each  

In [ ]:
# Control loop to track a spectral peak near a specified frequency until stopped. 

In [ ]:
# Run feedback process that attempts to iteratively target a spectral peak near a specified frequency. The feedback process will attempt to maximize the amplitude of the spectral peak by adjusting the phase and frequency of the IQ module's internal signal generator. The feedback process will run for a specified number of iterations, and will return the final frequency and phase settings that maximize the spectral peak amplitude.